In [35]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, PredefinedSplit
import joblib

In [16]:
df_base = pd.read_csv("baseline_data_cleaned.csv", index_col="oid", parse_dates=["veto_date"])
folds = pd.read_csv("stratified_folds.csv", index_col="oid")

In [17]:
features = [
    "n_points_alert","mag_min_alert","mag_max_alert",
    "mag_p05_alert","mag_p25_alert","mag_p50_alert","mag_p75_alert","mag_p95_alert",
    "excess_variance_alert","n_points_dr","mag_min_dr","mag_max_dr",
    "mag_p05_dr","mag_p25_dr","mag_p50_dr","mag_p75_dr","mag_p95_dr",
    "excess_variance_dr","distance_delight","hostsize"
]
X = df_base[features]
y = df_base["label"]

In [18]:
# Construimos los folds de validacion a partid del csv predefinido
test_fold = np.full(len(df_base), -1, dtype=int)
for i in range(1,6):
    mask = folds[f"val_fold{i}"] == 1
    test_fold[mask.values] = i-1

In [19]:
ps = PredefinedSplit(test_fold=test_fold)

In [20]:
# GridSeach parametros
param_grid = {
    "n_estimators":      [100, 200, 500],
    "max_depth":         [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "max_features":      ["sqrt", "log2"]
}

In [21]:
# Modelo RandomForest
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

In [22]:
# GridSearchCV
grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=ps,
    scoring="accuracy",
    verbose=2
)

In [23]:
# Buscamos hyperparametros
grid.fit(X, y)

print("Mejores parámetros encontrados:")
print(grid.best_params_)
print("Mejor accuracy (CV):", grid.best_score_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=   2.3s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=   2.6s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=   3.2s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=   2.2s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=   2.2s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=200; total time=   4.5s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=200; total time=   5.3s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=200; total time=   4.5s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=200; total time=   4.5s
[CV] END max_depth=N

In [24]:
# Evaluamos sobre cada fold
best_rf = grid.best_estimator_
for i in range(1,6):
    train_mask = folds[f"train_fold{i}"] == 1
    val_mask   = folds[f"val_fold{i}"]   == 1

    X_train, y_train = X[train_mask.values], y[train_mask.values]
    X_val,   y_val   = X[val_mask.values],   y[val_mask.values]

    y_pred = best_rf.fit(X_train, y_train).predict(X_val)
    print(f"\n=== Fold {i} ===")
    print(classification_report(y_val, y_pred))


=== Fold 1 ===
              precision    recall  f1-score   support

         bad       0.68      0.38      0.49       991
        good       0.76      0.92      0.83      2166

    accuracy                           0.75      3157
   macro avg       0.72      0.65      0.66      3157
weighted avg       0.74      0.75      0.72      3157


=== Fold 2 ===
              precision    recall  f1-score   support

         bad       0.69      0.42      0.53       988
        good       0.78      0.92      0.84      2169

    accuracy                           0.76      3157
   macro avg       0.74      0.67      0.68      3157
weighted avg       0.75      0.76      0.74      3157


=== Fold 3 ===
              precision    recall  f1-score   support

         bad       0.69      0.42      0.52      1002
        good       0.77      0.91      0.84      2155

    accuracy                           0.76      3157
   macro avg       0.73      0.67      0.68      3157
weighted avg       0.75   

In [26]:
param_grid_refined = {
    'n_estimators':      [400, 500, 600],
    'max_depth':         [8, 10, 12],
    'min_samples_split': [8, 10, 12],
    'max_features':      ['sqrt']
}

grid_refined = GridSearchCV(
    estimator=best_rf,
    param_grid=param_grid_refined,
    cv=ps,
    verbose=2,
    scoring='accuracy'
)

In [27]:
grid_refined.fit(X, y)

print("Mejores parámetros encontrados:")
print(grid_refined.best_params_)
print("Mejor accuracy (CV):", grid_refined.best_score_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
[CV] END max_depth=8, max_features=sqrt, min_samples_split=8, n_estimators=400; total time=   5.3s
[CV] END max_depth=8, max_features=sqrt, min_samples_split=8, n_estimators=400; total time=   6.4s
[CV] END max_depth=8, max_features=sqrt, min_samples_split=8, n_estimators=400; total time=   6.2s
[CV] END max_depth=8, max_features=sqrt, min_samples_split=8, n_estimators=400; total time=   4.9s
[CV] END max_depth=8, max_features=sqrt, min_samples_split=8, n_estimators=400; total time=   5.8s
[CV] END max_depth=8, max_features=sqrt, min_samples_split=8, n_estimators=500; total time=   6.7s
[CV] END max_depth=8, max_features=sqrt, min_samples_split=8, n_estimators=500; total time=   7.2s
[CV] END max_depth=8, max_features=sqrt, min_samples_split=8, n_estimators=500; total time=   7.3s
[CV] END max_depth=8, max_features=sqrt, min_samples_split=8, n_estimators=500; total time=   6.9s
[CV] END max_depth=8, max_features=sqrt, min_sa

In [28]:
# Entrenamiento final con los mejores parámetros
df = pd.read_csv("baseline_data_cleaned.csv", index_col="oid", parse_dates=["veto_date"])
folds = pd.read_csv("stratified_folds.csv", index_col="oid")

In [29]:
features = [
    "n_points_alert","mag_min_alert","mag_max_alert",
    "mag_p05_alert","mag_p25_alert","mag_p50_alert","mag_p75_alert","mag_p95_alert",
    "excess_variance_alert","n_points_dr","mag_min_dr","mag_max_dr",
    "mag_p05_dr","mag_p25_dr","mag_p50_dr","mag_p75_dr","mag_p95_dr",
    "excess_variance_dr","distance_delight","hostsize"
]
X = df[features]
y = df["label"]

In [30]:
train_mask = folds["test"] == 0
test_mask  = folds["test"] == 1

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

In [ ]:
best_params = {
    'n_estimators': 500,
    'max_depth': 12,
    'min_samples_split': 12,
    'max_features': 'sqrt',
    'random_state': 42,
    'n_jobs': -1
}
final_rf = RandomForestClassifier(**best_params)

In [32]:
final_rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=12, min_samples_split=12, n_estimators=500,
                       n_jobs=-1, random_state=42)

In [39]:
# Guardamos el modelo
joblib.dump(final_rf, "baseline_v1.joblib")

['baseline_v1.joblib']

In [ ]:
y_pred = final_rf.predict(X_test)
print("=== Classification Report on Test Set ===")
print(classification_report(y_test, y_pred, zero_division=0))
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

=== Classification Report on Test Set ===
              precision    recall  f1-score   support

         bad       0.69      0.43      0.53      1243
        good       0.78      0.91      0.84      2703

    accuracy                           0.76      3946
   macro avg       0.73      0.67      0.68      3946
weighted avg       0.75      0.76      0.74      3946

=== Confusion Matrix ===
[[ 530  713]
 [ 241 2462]]


In [38]:
confusion_matrix(y_test, y_pred)

array([[ 530,  713],
       [ 241, 2462]], dtype=int64)